# Lesson 3: pandas Internals

**Week 3 · Data Engineering Course**

---

pandas is built on numpy. Understanding what numpy does underneath — contiguous typed memory, vectorized C operations, the copy/view distinction — determines whether you use pandas correctly or produce code that is both slow and subtly wrong.

This lesson covers how pandas stores data, what operations are fast and why, what SettingWithCopyWarning means and how to avoid it, how to read large files without loading them all, and advanced reshaping and aggregation.

**Sections:**
1. The pandas Data Model
2. The Copy/View Problem
3. Vectorized Operations vs `.apply()`
4. Reading Files at Scale
5. Advanced String and Date Operations
6. Reshaping Data
7. Method Chaining with `.pipe()` and `.assign()`

In [ ]:
import pandas as pd
import numpy as np
import tracemalloc
import timeit
from pathlib import Path

RAW = Path('../data/raw')
CLEAN = Path('../data/clean')
CLEAN.mkdir(parents=True, exist_ok=True)

print(f'pandas {pd.__version__}  numpy {np.__version__}')

---

## 3.1 The pandas Data Model

A **DataFrame** is a dict-like container where each column is a **Series**, and each Series wraps a **numpy ndarray**.

Numpy stores data in **contiguous blocks of typed memory** — a flat array of 8-byte floats, or 8-byte integers, or 1-byte booleans. Operations on these arrays are implemented in C and loop at C speed, not Python speed.

When a column has dtype `object`, pandas stores Python object pointers instead of raw values. Each pointer is 8 bytes, but it points to a Python object in the heap that typically costs 28–56 bytes more. This is why `object` dtype is expensive for large string columns.

**dtype memory costs:**

| dtype | bytes per element | notes |
|-------|-------------------|-------|
| `bool` | 1 | |
| `int8` | 1 | max ±127 |
| `int64` | 8 | default for integers |
| `float32` | 4 | half the size of float64, acceptable for many prices |
| `float64` | 8 | default for floats |
| `object` | 8 (pointer) + object overhead | strings, mixed types — expensive |

In [ ]:
# Load Q1 and inspect what pandas inferred about dtypes
df = pd.read_csv(RAW / 'sales_q1.csv')

print('Inferred dtypes:')
print(df.dtypes)
print()
print('Memory usage (shallow):')
print(df.memory_usage())
print(f'Total (shallow): {df.memory_usage().sum() / 1024:.1f} KB')
print()
print('Memory usage (deep — includes actual string content):')
print(df.memory_usage(deep=True))
print(f'Total (deep): {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
# The difference between shallow and deep reveals the true cost of object columns

In [ ]:
# Prove that a Series is backed by a numpy array
quantity_series = df['quantity']
print(f'type of series:       {type(quantity_series)}')
print(f'dtype of series:      {quantity_series.dtype}')
print(f'underlying array:     {type(quantity_series.values)}')
print(f'values:               {quantity_series.values}')
print()

# int64 column: 8 bytes per element
print(f'8 elements * 8 bytes = {8 * 8} bytes for the quantity column')
print(f'actual size: {quantity_series.memory_usage()} bytes (includes index overhead)')

# object column: 8-byte pointer + heap object
category_series = df['category']
print(f'\ncategory dtype: {category_series.dtype}')
print(f'category memory (shallow): {category_series.memory_usage()} bytes')
print(f'category memory (deep):    {category_series.memory_usage(deep=True)} bytes')
print(f'heap overhead per element: {(category_series.memory_usage(deep=True) - category_series.memory_usage()) / len(category_series):.0f} bytes/element')

In [ ]:
# Categorical dtype: replace string objects with integer codes
# Stores each unique string ONCE, then uses small integers to refer to them
# Huge memory saving for low-cardinality columns (columns with few unique values)

df_cat = df.copy()
df_cat['category'] = df_cat['category'].astype('category')

print(f'category as object:      {df["category"].memory_usage(deep=True)} bytes')
print(f'category as categorical: {df_cat["category"].memory_usage(deep=True)} bytes')
print()
print(f'Unique values (categories): {df_cat["category"].cat.categories.tolist()}')
print(f'Underlying codes (int8):    {df_cat["category"].cat.codes.values}')
print()
# Each code is a tiny integer pointing into the categories array
# With 3 unique categories in 26 rows, this saves significant memory
# The benefit scales: a 10M-row DataFrame with 5 unique categories saves ~90% on that column
print('Rule: use categorical for any string column with < 50 unique values.')

---

## 3.2 The Copy/View Problem

When pandas returns data, it sometimes returns a **view** (a window into the original array — no copy made, changes reflect back) and sometimes a **copy** (an independent array — changes do not affect the original).

The rule is complex and inconsistent across pandas versions. The practical consequence: if you try to assign values to a subset of a DataFrame obtained through chained indexing, you may be modifying a temporary copy rather than the original. pandas warns about this with `SettingWithCopyWarning`.

**The safe pattern:** use `.loc[mask, column] = value` on the DataFrame directly, or call `.copy()` explicitly when you want an independent copy.

In [ ]:
# When does pandas return a view vs a copy?

df = pd.read_csv(RAW / 'sales_q1.csv')

# Single column selection: may return a view
col = df['category']
print(f'Is category a view? _is_copy attr exists: {hasattr(col, "_is_copy")}')
print(f'shares memory with df: {np.shares_memory(col.values, df["category"].values)}')
# True: modifying col would affect df (in older pandas)

# Boolean indexing: always returns a copy
mask = df['quantity'] > 1
subset = df[mask]
print(f'\nBoolean subset shares memory: {np.shares_memory(subset["quantity"].values, df["quantity"].values)}')
# False: subset is a copy, safe to modify independently

# .copy() — explicit, always safe
safe_copy = df[mask].copy()
print(f'Explicit copy shares memory: {np.shares_memory(safe_copy["quantity"].values, df["quantity"].values)}')

In [ ]:
# SettingWithCopyWarning: the most common pandas mistake
import warnings

df = pd.read_csv(RAW / 'sales_q1.csv')

# WRONG: chained indexing — modifying what may be a copy
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    # This might silently fail to modify df, or might work, depending on context
    df[df['quantity'] < 0]['category'] = 'INVALID'   # SettingWithCopyWarning
    if w:
        print(f'Warning raised: {w[0].category.__name__}')
        print(f'  {w[0].message}')

# CORRECT: .loc with a boolean mask modifies the DataFrame in place safely
df_clean = df.copy()
neg_mask = df_clean['quantity'].astype(str).str.strip().apply(
    lambda x: float(x) < 0 if x.lstrip('-').replace('.', '').isdigit() else False
)
df_clean.loc[neg_mask, 'category'] = 'INVALID'
print(f'\nRows marked INVALID: {(df_clean["category"] == "INVALID").sum()}')

In [ ]:
# The clean pattern: always work on a copy when modifying
# Never assume that df[condition] gives you something you can write to

df = pd.read_csv(RAW / 'sales_q1.csv')

# Pattern: take a copy of the full DataFrame, then modify it
work = df.copy()    # independent copy — all assignments are safe

# Now .loc assignments on work are unambiguously correct
work['category'] = work['category'].str.strip().str.title()
work.loc[work['category'] == 'Home & Kitchen', 'region'] = work.loc[work['category'] == 'Home & Kitchen', 'region'].fillna('Unknown')

print('Modified work DataFrame:')
print(work[['order_id', 'category', 'region']].head(5))
print(f'\nOriginal df category[0]: {df["category"].iloc[0]!r}  (unchanged)')
print(f'work category[0]:        {work["category"].iloc[0]!r}')

---

## 3.3 Vectorized Operations vs `.apply()`

Numpy vectorized operations loop in C over the underlying array. They run at C speed, with no Python overhead per element.

`.apply(func)` exits Python for every element, calls `func` as a Python function, and returns a Python object. For 1 million rows, this means 1 million Python function calls.

The performance gap is typically 10–100x. For a pipeline processing millions of rows, the wrong choice here is the single biggest performance bottleneck.

In [ ]:
# Build a large synthetic DataFrame to make the timing meaningful
np.random.seed(42)
N = 200_000
large = pd.DataFrame({
    'category': np.random.choice(['  electronics  ', 'CLOTHING', '  home & kitchen  '], N),
    'price_str': ['$' + str(round(np.random.uniform(10, 1500), 2)) for _ in range(N)],
    'quantity':  np.random.randint(1, 10, N).astype(float),
})
print(f'DataFrame shape: {large.shape}')
print(large.head(3))

In [ ]:
# Comparison 1: normalise category strings
# .str methods: vectorized (C speed)
# .apply(str.strip): Python call per element

def time_it(stmt, setup, n=3):
    t = timeit.timeit(stmt, setup=setup, number=n)
    return t / n

setup = 'import pandas as pd; import numpy as np; cat = large["category"].copy()'
setup_with_large = f'large = __import__("pandas").read_csv("../data/raw/sales_q1.csv"); cat = large["category"]'

# Run directly (not using timeit for simplicity)
import time

cat = large['category'].copy()

start = time.perf_counter()
result_vectorized = cat.str.strip().str.lower()
t_vec = time.perf_counter() - start

start = time.perf_counter()
result_apply = cat.apply(lambda x: str(x).strip().lower())
t_apply = time.perf_counter() - start

print(f'Vectorized .str.strip().str.lower(): {t_vec * 1000:.1f} ms')
print(f'.apply(lambda):                      {t_apply * 1000:.1f} ms')
print(f'Speedup: {t_apply / t_vec:.1f}x')
print(f'Results identical: {result_vectorized.equals(result_apply)}')

In [ ]:
# Comparison 2: clean a price string column
# Vectorized: .str.replace() + pd.to_numeric()
# apply: Python function per element

price = large['price_str'].copy()

start = time.perf_counter()
vec_result = pd.to_numeric(price.str.replace('$', '', regex=False), errors='coerce')
t_vec = time.perf_counter() - start

def clean_price_py(s):
    return float(str(s).replace('$', '').strip())

start = time.perf_counter()
apply_result = price.apply(clean_price_py)
t_apply = time.perf_counter() - start

print(f'Vectorized: {t_vec * 1000:.1f} ms')
print(f'apply:      {t_apply * 1000:.1f} ms')
print(f'Speedup: {t_apply / t_vec:.1f}x')
print()
print('Rule: if there is a .str method or pd.to_numeric/pd.to_datetime, use it.')
print('Use .apply() only when the logic cannot be expressed with vectorized operations.')

In [ ]:
# When .apply() IS the right choice:
# Row-wise operations that cannot be expressed column-by-column

df = pd.read_csv(RAW / 'sales_q1.csv')
df['unit_price'] = pd.to_numeric(
    df['unit_price'].astype(str).str.replace('$', '', regex=False),
    errors='coerce'
)
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['total']    = pd.to_numeric(df['total'],    errors='coerce')

# Validation that touches multiple columns in the same row:
# No vectorized equivalent exists for this kind of multi-column row check
def validate_row(row):
    qty = row['quantity']
    price = row['unit_price']
    total = row['total']
    if pd.isna(qty) or qty <= 0:
        return 'invalid_quantity'
    if pd.isna(price) or price < 0:
        return 'invalid_price'
    if pd.notna(total) and abs(total - qty * price) > 0.01:
        return 'wrong_total'
    return 'ok'

df['validation'] = df.apply(validate_row, axis=1)  # axis=1: apply row-wise
print(df['validation'].value_counts())

In [ ]:
# .assign(): create or transform columns in a chainable way
# Unlike df['col'] = ..., .assign() returns a new DataFrame

df = pd.read_csv(RAW / 'sales_q1.csv')

# Chain multiple transformations without intermediate assignments
clean = (
    df
    .assign(
        category   = lambda d: d['category'].str.strip().str.title(),
        product    = lambda d: d['product'].str.strip(),
        unit_price = lambda d: pd.to_numeric(
            d['unit_price'].astype(str).str.replace('$', '', regex=False),
            errors='coerce'
        ),
        quantity   = lambda d: pd.to_numeric(d['quantity'], errors='coerce'),
    )
    .assign(
        total = lambda d: d['total'].pipe(
            pd.to_numeric, errors='coerce'
        ).fillna(d['quantity'] * d['unit_price'])
    )
    .dropna(subset=['unit_price', 'quantity'])
    .loc[lambda d: d['quantity'] > 0]
)
print(f'Clean rows: {len(clean)}')
print(clean[['order_id', 'category', 'unit_price', 'quantity', 'total']].head(5))

---

## 3.4 Reading Files at Scale

`pd.read_csv()` has parameters that dramatically change how much memory it uses and how fast it runs. Using all of them together is the difference between a pipeline that works on a laptop and one that works in production.

In [ ]:
# Parameter 1: dtype — tell pandas what type each column is
# Without dtype, pandas reads everything as object first, then infers
# With dtype, pandas reads directly into the right type

tracemalloc.start()
tracemalloc.reset_peak()
df_auto = pd.read_csv(RAW / 'sales_q1.csv')
_, peak_auto = tracemalloc.get_traced_memory()

tracemalloc.reset_peak()
df_typed = pd.read_csv(
    RAW / 'sales_q1.csv',
    dtype={
        'order_id':   'int32',
        'quantity':   'float32',   # will be cleaned to int later
        'unit_price': 'object',    # has $ signs, needs string cleaning first
        'total':      'object',    # may be empty, needs string cleaning
    }
)
_, peak_typed = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f'Auto-inferred dtypes:  peak {peak_auto / 1024:.1f} KB')
print(f'Explicit dtype:        peak {peak_typed / 1024:.1f} KB')
print(f'\nauto dtypes:  {df_auto.dtypes.to_dict()}')
print(f'typed dtypes: {df_typed.dtypes.to_dict()}')

In [ ]:
# Parameter 2: usecols — read only the columns you need
# If your pipeline uses 5 of 20 columns, there is no reason to read the other 15

df_all_cols = pd.read_csv(RAW / 'sales_q1.csv')
df_selected = pd.read_csv(
    RAW / 'sales_q1.csv',
    usecols=['order_id', 'date', 'product', 'category', 'quantity', 'unit_price', 'total']
)

print(f'All columns:    {df_all_cols.shape}  {df_all_cols.memory_usage(deep=True).sum() / 1024:.1f} KB')
print(f'7 columns only: {df_selected.shape}  {df_selected.memory_usage(deep=True).sum() / 1024:.1f} KB')

In [ ]:
# Parameter 3: chunksize — read a large file in pieces
# pd.read_csv() with chunksize returns a TextFileReader iterator
# Each iteration yields a DataFrame of that many rows

# Simulate processing a large file by processing Q1 in chunks of 10 rows
chunk_iter = pd.read_csv(RAW / 'sales_q1.csv', chunksize=10)
print(f'Type: {type(chunk_iter)}')

revenue_by_cat = {}   # accumulate results across chunks

for i, chunk in enumerate(chunk_iter):
    # Clean the chunk
    chunk['unit_price'] = pd.to_numeric(
        chunk['unit_price'].astype(str).str.replace('$', '', regex=False),
        errors='coerce'
    )
    chunk['quantity'] = pd.to_numeric(chunk['quantity'], errors='coerce').fillna(0)
    chunk['total'] = chunk['quantity'] * chunk['unit_price']
    chunk['category'] = chunk['category'].str.strip().str.title()

    # Aggregate within chunk
    for cat, grp in chunk.groupby('category'):
        revenue_by_cat[cat] = revenue_by_cat.get(cat, 0.0) + grp['total'].sum()

    print(f'  chunk {i}: {len(chunk)} rows  memory in use: ~{chunk.memory_usage(deep=True).sum() / 1024:.1f} KB')

print(f'\nRevenue by category (accumulated across chunks):')
for cat, rev in sorted(revenue_by_cat.items(), key=lambda x: -x[1]):
    print(f'  {cat}: ${rev:,.2f}')

---

## 3.5 Advanced String and Date Operations

`.str` accessor methods are vectorized — they run in C on the underlying object array, which makes them significantly faster than `.apply()`. They also compose cleanly without loops.

For dates, specifying `format=` in `pd.to_datetime()` is critical for large datasets. Without it, pandas tries multiple format patterns for each value. With it, it uses a direct C-speed parser.

In [ ]:
# .str.extract() with named capture groups
# Returns a DataFrame with one column per capture group
# This is the vectorized way to split structured strings

import pandas as pd

dates = pd.Series(['2023-07-14', '2023-08-05', 'not-a-date', '2024-01-01'])

extracted = dates.str.extract(r'(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})')
print(extracted)
print(f'\ndtype of year column: {extracted["year"].dtype}')
# Extracted groups are object dtype (strings) — convert if needed
extracted['year'] = pd.to_numeric(extracted['year'])
extracted['month'] = pd.to_numeric(extracted['month'])
print(extracted)
print(f'\ndtype after conversion: {extracted["year"].dtype}')

In [ ]:
# pd.to_datetime(): with and without format=
# Without format: pandas tries to infer, slow on large datasets
# With format: direct C-speed parsing

dates_series = pd.Series(['2023-07-14', '2023-08-05', '2023-09-01'] * 10_000)

start = time.perf_counter()
dates_inferred = pd.to_datetime(dates_series, infer_datetime_format=True)
t_infer = time.perf_counter() - start

import time
start = time.perf_counter()
dates_format = pd.to_datetime(dates_series, format='%Y-%m-%d')
t_format = time.perf_counter() - start

print(f'Without format (infer): {t_infer * 1000:.1f} ms')
print(f'With format=:           {t_format * 1000:.1f} ms')
print(f'Speedup: {t_infer / t_format:.1f}x')
print()
print(f'Result dtype: {dates_format.dtype}')   # datetime64[ns]

In [ ]:
# Handling multiple date formats in one column
# The sales data has 2023-01-05, 15/01/2023, Jan 20 2023 in Q1
# Strategy: try each format with errors='coerce', then combine

df = pd.read_csv(RAW / 'sales_q1.csv')
raw_dates = df['date'].copy()
print('Raw dates sample:')
print(raw_dates.head(10).tolist())

formats = ['%Y-%m-%d', '%Y/%m/%d', '%d/%m/%Y', '%b %d %Y']
parsed = pd.Series([pd.NaT] * len(raw_dates))

for fmt in formats:
    still_null = parsed.isna()
    if not still_null.any():
        break
    attempt = pd.to_datetime(raw_dates[still_null].str.strip(), format=fmt, errors='coerce')
    parsed[still_null] = attempt

print(f'\nParsed dates:')
print(parsed.head(10).tolist())
print(f'Null count: {parsed.isna().sum()}')

df['date_parsed'] = parsed
print(df[['date', 'date_parsed']].dropna().head(8))

In [ ]:
# pd.cut(): bin numeric values into categories
# Useful for creating revenue tiers, age groups, quantity buckets

df = pd.read_csv(RAW / 'sales_q2.csv')
df['total'] = pd.to_numeric(df['total'], errors='coerce')

bins   = [0, 100, 300, 1000, float('inf')]
labels = ['Low', 'Medium', 'High', 'Premium']

df['revenue_tier'] = pd.cut(df['total'], bins=bins, labels=labels)

print('Revenue tier distribution:')
print(df['revenue_tier'].value_counts().sort_index())
print()
print(df[['order_id', 'total', 'revenue_tier']].head(8))
print(f'\ndtype: {df["revenue_tier"].dtype}')   # CategoricalDtype

---

## 3.6 Reshaping Data

Data comes in two forms:
- **Wide format**: one row per entity, multiple value columns (`jan_revenue`, `feb_revenue`, `mar_revenue`)
- **Long format**: one row per observation (`entity`, `month`, `revenue`)

Most analysis and visualisation tools prefer long format. Databases expect long format. `.melt()` converts wide to long. `.pivot_table()` converts long to wide with aggregation.

In [ ]:
# Load and prepare data
import pandas as pd

df_q1 = pd.read_csv(RAW / 'sales_q1.csv')
df_q2 = pd.read_csv(RAW / 'sales_q2.csv')
df_q3 = pd.read_csv(RAW / 'sales_q3.csv')

for df in [df_q1, df_q2, df_q3]:
    df['category'] = df['category'].str.strip().str.title()
    df['total']    = pd.to_numeric(df['total'],    errors='coerce')
    df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')

all_data = pd.concat([df_q1, df_q2, df_q3], ignore_index=True)

# Build a wide summary: categories as rows, quarters as columns
all_data['quarter'] = 'Q' + (
    pd.to_numeric(all_data['order_id'].astype(str).str[0])
).astype(str)

wide = all_data.groupby(['category', 'quarter'])['total'].sum().unstack(fill_value=0).round(2)
print('Wide format (categories x quarters):')
print(wide)

In [ ]:
# .melt(): wide -> long
long = wide.reset_index().melt(
    id_vars='category',
    value_vars=['Q1', 'Q2', 'Q3'],
    var_name='quarter',
    value_name='revenue'
)
print('Long format after melt:')
print(long.sort_values(['category', 'quarter']).reset_index(drop=True))

In [ ]:
# .pivot_table(): long -> wide with aggregation
# More flexible than .unstack() — handles multiple values, multiple aggfuncs

pivot = all_data.pivot_table(
    index='category',
    columns='quarter',
    values='total',
    aggfunc='sum',
    fill_value=0
).round(2)

print('Pivot table (equivalent to the wide format above):')
print(pivot)

# Multiple aggregation functions at once
multi_agg = all_data.pivot_table(
    index='category',
    columns='quarter',
    values='total',
    aggfunc=['sum', 'count'],
    fill_value=0
)
print('\nMulti-agg pivot (sum and count):')
print(multi_agg)

---

## 3.7 Method Chaining with `.pipe()`

`.assign()` lets you add or transform columns inline. `.loc[lambda d: ...]` lets you filter inline. `.pipe(func)` lets you call any function inline — even one that takes a DataFrame as its first argument.

Together, these let you write a full cleaning pipeline as a single expression. Every transformation is named and in sequence. You can insert `.pipe(log_shape)` anywhere to debug without breaking the chain.

In [ ]:
# .pipe() debugging helper — insert anywhere to print state without breaking the chain
def log_shape(df, label=''):
    print(f'[{label}] shape={df.shape}  nulls={df.isnull().sum().sum()}')
    return df   # must return the DataFrame for the chain to continue

def drop_duplicates_on(df, key):
    '''Drop duplicate rows based on key column.'''
    return df.drop_duplicates(subset=[key])

def standardise_category(df, typo_map):
    '''Normalise category and apply typo corrections.'''
    df = df.copy()
    df['category'] = df['category'].str.strip().str.title().replace(typo_map)
    return df

TYPO_MAP = {'Electrnics': 'Electronics', 'Home & Kithen': 'Home & Kitchen'}

# Full cleaning pipeline as one expression
cleaned = (
    pd.read_csv(RAW / 'sales_q1.csv')
    .pipe(log_shape, 'after read')
    .pipe(drop_duplicates_on, key='order_id')
    .pipe(log_shape, 'after dedup')
    .assign(
        product    = lambda d: d['product'].str.strip(),
        unit_price = lambda d: pd.to_numeric(
            d['unit_price'].astype(str).str.replace('$', '', regex=False), errors='coerce'
        ),
        quantity   = lambda d: pd.to_numeric(d['quantity'], errors='coerce'),
        total      = lambda d: pd.to_numeric(d['total'], errors='coerce'),
    )
    .assign(
        total = lambda d: d['total'].fillna(d['quantity'] * d['unit_price'])
    )
    .pipe(standardise_category, TYPO_MAP)
    .loc[lambda d: d['quantity'].notna() & (d['quantity'] > 0)]
    .pipe(log_shape, 'final')
    .reset_index(drop=True)
)

print(f'\nCleaned shape: {cleaned.shape}')
print(cleaned[['order_id', 'category', 'unit_price', 'quantity', 'total']].head(5))

In [ ]:
# .transform() vs .agg(): understanding the difference
# .agg() reduces a group to a scalar — the result has fewer rows than the input
# .transform() computes a group aggregate but broadcasts it back to every row

df = cleaned.copy()

# agg: one row per category
cat_revenue = df.groupby('category')['total'].agg('sum')
print('agg result (one row per category):')
print(cat_revenue)
print()

# transform: one row per original row, with the group total broadcast
df['category_revenue'] = df.groupby('category')['total'].transform('sum')
df['pct_of_category'] = (df['total'] / df['category_revenue'] * 100).round(1)
print('transform result (row-level pct of category total):')
print(df[['order_id', 'product', 'category', 'total', 'category_revenue', 'pct_of_category']].head(8))

---

## Summary

| Concept | What to remember |
|---------|------------------|
| dtype and memory | `object` dtype costs 28–56 bytes per element on the heap. Use `category` for low-cardinality strings and explicit `dtype=` in `read_csv`. |
| Views vs copies | Boolean indexing returns a copy. Single column access may return a view. Always call `.copy()` when you need independence, and use `.loc[mask, col] = value` for in-place assignment. |
| SettingWithCopyWarning | It means you tried to assign via chained indexing. Fix: work on `.copy()` of the DataFrame, or use `.loc`. |
| Vectorized vs `.apply()` | `.str` methods and numpy operations are 10–100x faster than `.apply()`. Use `.apply(func, axis=1)` only for logic that genuinely requires multiple columns. |
| `.assign()` and `.pipe()` | Write cleaning as a single method chain. Insert `.pipe(log_shape)` for debugging. Each `.assign()` returns a new DataFrame. |
| `chunksize` | Required for files larger than available RAM. Accumulate aggregates across chunks; never `pd.concat` all chunks into one list. |
| `pd.to_datetime(format=)` | Specify `format=` for C-speed parsing. Looping over formats with `errors='coerce'` handles multiple date styles efficiently. |
| `.transform()` vs `.agg()` | `agg` reduces. `transform` broadcasts the group result back to each original row. |